# M1 · Fine-tuning con LoRA — triaje de acciones constitucionales (Colombia)

Produce: adaptador LoRA en `models/m1-adaptador-<id>`, scorecards base vs ajustado,
y `results/m1_predicciones.json`, que es lo que consume M3.

Requiere GPU (A100).

**Datos:** dataset v3.5.0 · 160 ejemplos · 123 train · 37 validation

Esa linea no esta escrita a mano dos veces: `data/auditar_notebooks.py` la lee
y la compara con `data/dataset.json`.

**La logica vive en `amparo/pipelines/m1_finetuning.py`**: este notebook solo
llama a sus etapas, una por celda, para ver la salida de cada una. El porque de
cada decision esta en los docstrings de esas funciones y en
`amparo/finetuning/` (`datos`, `generacion`, `entrenamiento`).

In [ ]:
# SIN `-U` Y CON LOS PINS DE `requirements.txt`: son las versiones que
# `m1_hiperparametros.json` registra de la corrida. `groq` hace falta para la
# seccion 7 (redaccion).
!pip install -q transformers==5.20.1 peft==0.21.2 accelerate==1.15.0 datasets==5.1.0 groq==1.7.0

# Colab trae torchao 0.10.0 y peft >=0.21 exige >0.16: al importar
# `get_peft_model` lanza ImportError con el modelo ya en VRAM. Aqui se entrena
# en bf16 y torchao solo sirve para cuantizar, asi que se desinstala.
!pip uninstall -y -q torchao

import importlib.util
assert importlib.util.find_spec("torchao") is None, (
    "torchao sigue instalado: Entorno -> Reiniciar sesion y corre esta celda de nuevo")

import torch
print("GPU :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "SIN GPU")
print("VRAM:", f"{torch.cuda.get_device_properties(0).total_memory/1e9:.0f} GB" if torch.cuda.is_available() else "-")
assert torch.cuda.is_available(), "Activa la GPU: Runtime -> Change runtime type -> A100"

## 0 · Proyecto y corrida

La primera celda monta Drive y pone la raiz del proyecto en `sys.path`: es lo
unico que no puede vivir en el paquete, porque hasta entonces `amparo` no se
puede importar. `m1.preparar()` hace el resto con `amparo.ejecucion.entorno`:
corrida nueva, sello de Drive, clave de Groq y `arranque.celda("m1")`.

In [ ]:
import sys
from pathlib import Path

try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    pass

CANDIDATAS = [Path("/content/drive/MyDrive/agenteAmparo"),
              Path("/content/drive/MyDrive/Colab Notebooks/agenteAmparo"),
              Path("/content/agenteAmparo"), Path("."), Path("..")]
RAIZ = next((p.resolve() for p in CANDIDATAS if (p / "amparo" / "nucleo" / "schema.py").exists()), None)
if RAIZ is None:
    raise SystemExit("No encuentro el proyecto (busque amparo/nucleo/schema.py en: "
                     + ", ".join(map(str, CANDIDATAS)) + "). Revisa que Drive este montado "
                     "y que la carpeta se llame agenteAmparo.")
sys.path.insert(0, str(RAIZ))

In [ ]:
from amparo.pipelines import m1_finetuning as m1

ctx = m1.preparar()

### 0.1 · Antes de la GPU: ¿sirvio el arreglo del extractor?

No es de M1: esta aqui porque es lo unico que puede ahorrar el ciclo entero,
antes de bajar 15 GB de pesos. `EL ARREGLO FUNCIONA` -> sigue; `SIGUEN FALLANDO N`
-> para y decide. No aborta: M1 no depende del extractor del agente.

In [ ]:
m1.validar_extractor(ctx)

## 1 · Por qué un decoder y no un clasificador

In [ ]:
m1.cobertura_clasificador(ctx)

## 2 · Qué decoder: fertilidad del tokenizador

In [ ]:
m1.fertilidad_tokenizadores(ctx)

## 3 · Modelo base y dataset v3.5.0

Siembra con los interruptores de kernel (`corridas.sembrar`) y carga
Qwen2.5-7B en bf16. Despues, los splits sin fuga hacia el eval set y la
tokenizacion con `MAX_LEN` medido (no adivinado).

In [ ]:
m1.cargar_modelo(ctx)

In [ ]:
m1.cargar_datos(ctx)

In [ ]:
m1.preparar_dataset(ctx)

## 4 · Baseline: el modelo sin ajustar

In [ ]:
m1.baseline(ctx)

### 4.1 · El baseline de FORMATO: al base se le enseña el esquema

Mismo modelo y misma generacion; lo unico que cambia es que el esquema va
demostrado en tres turnos (del train, nunca del eval). Responde cuanto del
salto era formato y cuanto razonamiento. Va antes de LoRA a proposito.

In [ ]:
m1.baseline_few_shot(ctx)

## 5 · Fine-tuning con LoRA

In [ ]:
m1.aplicar_lora(ctx)

In [ ]:
m1.configurar_entrenamiento(ctx)

In [ ]:
m1.entrenar(ctx)

In [ ]:
m1.curva_de_perdida(ctx)

## 6 · ¿Mejoró?

In [ ]:
m1.evaluar_ajustado(ctx)

In [ ]:
m1.comparar_con_base(ctx)

### 6.1 · Abstención

Se mide en validation, que trae una de cada familia: el desglose dice CUAL
falla. Las dos cifras (`acierto_abstencion` y `abstencion_indebida`) van juntas.

In [ ]:
m1.abstencion_validation(ctx)

### 6.3 · Quién decide el mecanismo: el base extrae, las reglas deciden

Sonda del adaptador como extractor (no sirve: ver `extractor` en el artefacto)
y el brazo con el adaptador apagado, guardado por un umbral de fallos de
extraccion del 10%.

In [ ]:
m1.brazo_hibrido(ctx)

## 7 · La capa de redacción — del bloque al lenguaje natural

Una capa de presentacion, nunca un decisor distinto: no añade normas ni
hechos, y el texto dice siempre la via y la procedencia. Sin `GROQ_API_KEY`
queda registrada como no medida.

In [ ]:
m1.redaccion(ctx)

## 8 · Artefactos

Lo que se guarda en la carpeta de la corrida no esta entregado hasta que se
copia a `results/`. Esta etapa publica los artefactos de M1 y versiona el
adaptador en `models/m1-adaptador-<id>`.

In [ ]:
m1.publicar_artefactos(ctx)

### 8.1 · Estabilidad entre corridas

Compara con la corrida previa de m1. La identidad solo prueba estabilidad con
los mismos datos y el mismo codigo; si difieren, separa las tres causas
(hardware, previa sin flags de determinismo, no-determinismo residual).

In [ ]:
m1.estabilidad(ctx)

In [ ]:
m1.cerrar(ctx)

## 9 · Qué mirar

1. **`acc_mecanismo` contra la mayoritaria** como liston minimo, y el **macro F1**
   para el delta contra el base: la accuracy sola premia colapsar en "tutela".
2. **Gold contra adversarial.** La brecha mide si razona o reconoce formas.
3. **Alucinacion en los dos ambitos** (`fundamento` y toda la salida).
4. **`eval_loss` por epoca.** Si toco minimo antes del final, sobraban epocas.
5. **Abstencion en validation**, en los dos sentidos y por familia.
6. **Estabilidad contra la corrida previa**, solo concluyente con `mismos_datos`.
7. **Degeneracion**: cuantas salidas se descarrilaron tras el cuarto campo.
   La cola se recorta antes de puntuar y el numero se reporta.